# 🔧 題目 4：叫車服務交通熱點
# Mini Data Pipeline 工作坊

> **情境**：你是叫車公司的資料顧問。營運主管想知道哪些時段地區叫車最多。
>
> **Pipeline**：`CSV → pandas → SQLite (raw/cleaned/analyzed) → SQL → LLM → FastAPI → Streamlit`
>
> **資料**：[NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)（2,000 筆取樣）
>
> 📄 詳細需求見 `requirements_spec.md`

---

### 📋 今日目標

| 必做（Section 1-8） | 回家作業（Section 9-10） |
|------|------|
| ✅ ETL pipeline（CSV → SQLite 三表） | ⭐ FastAPI API |
| ✅ SQL 查詢統計分析 | ⭐ ipywidgets / Streamlit Dashboard |
| ✅ LLM 分類分析 | ⭐ 本地部署 |
| ✅ 顧問報告 output/report.md | |

### 🗺️ 標記說明

| 標記 | 意思 |
|------|------|
| `🟢 簡單` | 開放式 — 提示裡有範例教語法，自己應用 |
| `🟡 中等` | 半骨架 — 用別的情境示範，你翻譯到自己的欄位 |
| `🔴 較難` | 完整骨架 — 結構都給了，填關鍵處 |
| `（不需要改）` | 直接跑 |


## Section 0：環境設定

直接跑，不需要改。


In [ ]:
# （不需要改）Colab 環境自動設定
import os
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('topic_4'):
        !git clone https://github.com/lu791019/midterm-mvp-template.git /content/repo
    os.chdir('/content/repo/data/raw/topic_4')
    print("✅ Colab：已設定工作目錄 =", os.getcwd())
else:
    print("✅ 本地環境，工作目錄 =", os.getcwd())


In [ ]:
# （不需要改）
import pandas as pd
import sqlite3
import os
import json
print('✅ 套件載入完成')


In [ ]:
# （不需要改）
OPENAI_API_KEY = ""
if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if line.startswith("OPENAI_API_KEY"):
                OPENAI_API_KEY = line.strip().split("=", 1)[1]
print("✅ API Key" if OPENAI_API_KEY else "⚠️ fallback 模式")


---
## Section 1：Extract — 讀取資料 + 寫入 raw 表

> pipeline 第一步：**資料進入系統**。


### Step 1-1 🟢 簡單：讀取 CSV

> 💡 `pd.read_csv()` 把 CSV 讀成 DataFrame
> 💡 **範例**：如果要讀外送訂單資料：
> ```python
> df = pd.read_csv("deliveries.csv")
> print(f"共 {len(df)} 筆")
> print(list(df.columns))
> df.head()
> ```
> 🎯 現在對 `trips.csv` 做同樣的事
> ✅ 預期：2,000 筆


In [ ]:
# TODO 🟢: 讀取 trips.csv

# 相關程式碼：
# df_raw = pd.read_csv("trips.csv")
# print(f"📊 {len(df_raw)} 筆, {len(df_raw.columns)} 欄")
# print(list(df_raw.columns))
# df_raw.head()


### Step 1-2 🟢 簡單：檢查資料品質

> 💡 **範例**：檢查外送訂單：
> ```python
> print(df.dtypes)
> print(df.isnull().sum())
> print(df.describe())
> ```
> 🎯 對 df_raw 做同樣三件事


In [ ]:
# TODO 🟢: 檢查品質

# 相關程式碼：
# print(df_raw.dtypes)
# print(df_raw.isnull().sum())
# print(df_raw.describe())


### Step 1-3 🟢 簡單：自由探索

> 💡 **範例**：
> ```python
> df["order_time"].value_counts().head(10)
> df["order_time"].nunique()
> df.sample(5)
> ```
> 🎯 用上面的招式探索你的資料


In [ ]:
# TODO 🟢: 自由探索

# 相關程式碼：
# df_raw["tpep_pickup_datetime"].value_counts().head(10)
# df_raw["tpep_pickup_datetime"].nunique()


### Step 1-4 🟡 中等：建立 SQLite + 寫入 raw 表

> 💡 **範例**：把外送訂單存進資料庫：
> ```python
> conn = sqlite3.connect("warehouse.db")
> df.to_sql("raw_inventory", conn, if_exists="replace", index=False)
> result = pd.read_sql("SELECT COUNT(*) as total FROM raw_inventory", conn)
> print(f"raw_inventory: {result['total'][0]} 筆")
> ```
> 🎯 建立 `pipeline.db`，寫入 `raw_trips` 表
> ✅ 預期：raw_trips: 2000 筆


In [ ]:
# TODO 🟡: 建立 SQLite，寫入 raw 表
DB_PATH = "pipeline.db"

# 相關程式碼：
# conn = sqlite3.connect(DB_PATH)
# df_raw.to_sql("raw_trips", conn, if_exists="replace", index=False)
# ...驗證筆數


---
## Section 2：Transform — 清洗 + 寫入 cleaned 表

> 從**資料庫**讀出 → 清洗 → 寫回資料庫。


### Step 2-1 🟢 簡單：從 raw 表讀出

> 💡 **範例**：`pd.read_sql("SELECT * FROM raw_inventory", conn)`
> ✅ 預期：df 有 2000 筆


In [ ]:
# TODO 🟢: 從 raw_trips 讀出


### Step 2-2 🟢 簡單：處理缺漏值

> 💡 **範例**：`df.dropna(subset=["product_name", "price"])`
> 🎯 刪除 `tpep_pickup_datetime` 和 `trip_distance` 為空的列


In [ ]:
# TODO 🟢: 刪除缺漏值


### Step 2-3 🟡 中等：日期轉換、提取時間特徵、計算行程時間、合併地名（taxi_zone_lookup.csv）、過濾異常值

> 💡 **範例**：假設有外送訂單，要提取時間和計算配送時間：
> ```python
> orders["order_time"] = pd.to_datetime(orders["order_time"])
> orders["hour"] = orders["order_time"].dt.hour
> orders["day_of_week"] = orders["order_time"].dt.day_name()
> orders["delivery_min"] = ((orders["delivered_time"] - orders["order_time"]).dt.total_seconds() / 60).round(1)
> ```
> 🎯 對你的資料：
> 1. pickup/dropoff datetime 轉日期，提取 hour, day_of_week
> 2. 計算 trip_duration_min
> 3. 過濾異常值（距離/金額 ≤ 0）
> 4. 用 `df.merge()` 合併 taxi_zone_lookup.csv 把 PULocationID 轉成地名
> ⚠️ merge 是這題的難點


In [ ]:
# TODO 🟡: 清洗轉換


### 🏁 清洗檢查點

> 直接跑。全部 ✅ 才往下。


In [ ]:
# （不需要改）
print(f"清洗前→清洗後: {len(df)} 筆")
print(f"缺漏值: {df.isnull().sum().sum()}")
assert df.isnull().sum().sum() == 0, "❌ 還有缺漏值"
print("✅ 檢查通過")


### Step 2-4 🟢 簡單：寫入 cleaned 表

> 💡 跟 Step 1-4 一樣，表名改 `cleaned_trips`


In [ ]:
# TODO 🟢: 寫入 cleaned_trips 表


---
## Section 3：SQL 統計分析

> 用 SQL 從資料庫查詢。


### Step 3-1 🟡 中等：上車熱點 Top 15

> 💡 **範例**：查各部門平均考績：
> ```python
> pd.read_sql("""
>     SELECT department, COUNT(*), ROUND(AVG(score), 2)
>     FROM employees GROUP BY department ORDER BY AVG(score) DESC
> """, conn)
> ```
> 🎯 從 `cleaned_trips` 查上車熱點 Top 15
> 💡 提示：GROUP BY pickup_borough, pickup_zone


In [ ]:
# TODO 🟡: 上車熱點 Top 15
stat1 = pd.read_sql("""

""", conn)
stat1


### Step 3-2 🟡 中等：尖峰時段統計

> 💡 提示：GROUP BY hour，看各時段叫車量和平均車資


In [ ]:
# TODO 🟡: 尖峰時段統計
stat2 = pd.read_sql("""

""", conn)
stat2


### Step 3-3 🟡 中等：視覺化

> 💡 **範例**：`df.plot.barh(x="department", y="avg_score", figsize=(10,5))`
> 🎯 把統計結果畫成至少一張圖


In [ ]:
# TODO 🟡: 視覺化
import matplotlib.pyplot as plt


### Step 3-4 🟢 簡單：自由探索 SQL

> 💡 靈感：小費和時段有關嗎？哪個星期幾最忙？


In [ ]:
# TODO 🟢: 你自己的 SQL


### Step 3-5 🟢 簡單：存統計結果

> 💡 `os.makedirs("processed", exist_ok=True)` + `df.to_csv()`


In [ ]:
# TODO 🟢: 存結果到 processed/


---
## Section 4：LLM 加值分析

> 對區域分類（商業區/住宅區/交通樞紐/觀光區/其他）。helper 函式已寫好，你要：呼叫、看結果、跑批次、寫入資料庫。


In [ ]:
import requests
def llm_analyze(text, api_key=None):
    if api_key: return _llm_api(text, api_key)
    return _llm_fallback(text)
def _llm_api(text, api_key):
    prompt = f"""請分析以下叫車熱點區域，回傳 JSON：
{{"area_type": "商業區/住宅區/交通樞紐/觀光區/其他", "insight": "一句話調度建議"}}\n文字：{text[:300]}"""
    try:
        resp = requests.post("https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}"},
            json={"model": "gpt-4o-mini", "messages": [{"role": "user", "content": prompt}], "temperature": 0.3}, timeout=30)
        content = resp.json()["choices"][0]["message"]["content"].strip()
        if content.startswith("```"): content = content.split("\n", 1)[1].rsplit("```", 1)[0]
        return json.loads(content)
    except: return _llm_fallback(text)
def _llm_fallback(text):
    t = text.lower()
    if any(w in t for w in ["airport","jfk","laguardia","newark"]): cat = "交通樞紐"
    elif any(w in t for w in ["midtown","financial","wall st","downtown"]): cat = "商業區"
    elif any(w in t for w in ["times square","central park","museum","theater"]): cat = "觀光區"
    elif any(w in t for w in ["heights","village","park slope","brooklyn"]): cat = "住宅區"
    else: cat = "其他"
    return {"area_type": cat, "insight": text[:50] + "..."}
print("✅ LLM Helper")


### Step 4-1 🟢 簡單：單筆測試

> 💡 **範例**：
> ```python
> test = df["product_name"].iloc[0]
> result = analyze(test)
> print(result)
> ```
> 🎯 取 df 的第一筆 `pickup_zone`，呼叫 `llm_analyze()`


In [ ]:
# TODO 🟢: 單筆測試


### Step 4-2 🟡 中等：批次分析

> 💡 **範例**：
> ```python
> results = []
> for i, row in df.head(50).iterrows():
>     r = analyze(row["product_name"])
>     results.append(r)
> ```
> 🎯 對 df 前 50 筆的 `pickup_zone` 批次呼叫


In [ ]:
# TODO 🟡: 批次分析
BATCH_SIZE = 50
api_key = OPENAI_API_KEY if OPENAI_API_KEY else None


### Step 4-3 🟡 中等：整理結果 + 寫入 analyzed 表

> 💡 **範例**：
> ```python
> df_analyzed = df.head(50).copy()
> df_analyzed["category"] = [r["category"] for r in results]
> df_analyzed.to_sql("analyzed_products", conn, if_exists="replace", index=False)
> ```
> 🎯 把 LLM 結果加回 df，寫入 `analyzed_trips` 表


In [ ]:
# TODO 🟡: 整理 + 寫入 analyzed_trips


---
## Section 5：驗證 pipeline


### Step 5-1 🟡 中等：跨表查詢

> 💡 **範例**：
> ```python
> pd.read_sql("""
>     SELECT 'raw' as layer, COUNT(*) as rows FROM raw_table
>     UNION ALL SELECT 'cleaned', COUNT(*) FROM cleaned_table
>     UNION ALL SELECT 'analyzed', COUNT(*) FROM analyzed_table
> """, conn)
> ```
> 🎯 查 `raw_trips`, `cleaned_trips`, `analyzed_trips` 三表


In [ ]:
# TODO 🟡: 跨表查詢
lineage = pd.read_sql("""

""", conn)
print(lineage.to_string(index=False))


---
## Section 6：產出報告


### Step 6-1 🟢 簡單：寫報告

> 💡 用 f-string 嵌入數字，存到 output/report.md
> 🎯 報告要有具體數字和建議


In [ ]:
# TODO 🟢: 寫報告
report = f"""# 叫車服務交通熱點分析報告

## 資料概要
（填入分析筆數、關鍵統計）

## 關鍵發現
（根據 Section 3 統計，寫 2-3 個發現）

## 建議
（寫 2-3 條有數據支撐的建議）

## Pipeline
CSV → pandas → SQLite → SQL → LLM → 本報告
"""
os.makedirs("output", exist_ok=True)
with open("output/report.md", "w") as f: f.write(report)
print("✅ report.md")


---
## Section 7：打包確認

> 直接跑。


In [ ]:
# （不需要改）
checks = [("pipeline.db","DB"), ("processed","統計"), ("output/report.md","報告")]
for p,d in checks: print(f"  {'✅' if os.path.exists(p) else '❌'} {d}: {p}")
if os.path.exists("pipeline.db"):
    c = sqlite3.connect("pipeline.db")
    for t in ["raw_trips","cleaned_trips","analyzed_trips"]:
        try: print(f"  ✅ {t}: {pd.read_sql(f'SELECT COUNT(*) as n FROM [{t}]', c)['n'][0]}")
        except: print(f"  ❌ {t}")
    c.close()
print("\n📋 接下來：README + upgrade_plan + 2 分鐘 Demo + Section 9-10（回家作業）")


---
## Section 8：FastAPI

> 🅰️ 在下面寫 / 🅱️ 開 `api.py`（solution）


### Step 8-1 🔴 較難：定義 endpoint

> 💡 **範例**：把考績做成 API：
> ```python
> from fastapi import FastAPI
> api = FastAPI(title="考績 API")
> @api.get("/top")
> def top():
>     c = sqlite3.connect("school.db")
>     df = pd.read_sql("SELECT name, score FROM employees ORDER BY score DESC LIMIT 10", c)
>     c.close()
>     return df.to_dict(orient="records")
> ```
> 🎯 定義 /health、/stats（熱點排行）、/analyzed


In [ ]:
# TODO 🔴: FastAPI
!pip install -q fastapi uvicorn nest_asyncio
from fastapi import FastAPI
import nest_asyncio
nest_asyncio.apply()

api = FastAPI(title="叫車服務交通熱點 API")

@api.get("/health")
def health():
    return {"status": "ok"}

# TODO: /stats


# TODO: /analyzed


print("✅ API 定義完成")


### Step 8-2 🟡 中等：啟動 + 測試


In [ ]:
# 啟動（直接跑）
import threading, uvicorn, time
thread = threading.Thread(target=uvicorn.run, args=(api,), kwargs={"host":"0.0.0.0","port":8000,"log_level":"warning"})
thread.daemon = True
thread.start()
time.sleep(2)
print("✅ API 已啟動")

# TODO 🟡: 用 requests 測試
import requests


> 🅱️ `api.py` 是 solution。本地：`uvicorn api:app --reload --port 8000`


---
## Section 9（回家作業）：Dashboard

> 🅰️ ipywidgets（Colab）/ 🅱️ `app.py`（Streamlit，solution）


### Step 9-1 🔴 較難：互動 Dashboard

> 💡 **範例**：選部門看薪資分佈：
> ```python
> import ipywidgets as widgets
> from IPython.display import display, clear_output
> dept_dd = widgets.Dropdown(options=["全部","工程部","業務部"], description="部門：")
> def update(dept):
>     clear_output(wait=True)
>     display(dept_dd)
>     data = df if dept == "全部" else df[df["department"] == dept]
>     print(f"{dept}: {len(data)} 人")
>     data["salary"].hist()
>     plt.show()
> widgets.interact(update, dept=dept_dd)
> ```
> 🎯 做一個「選 pickup_borough → 看 total_amount 分佈」的互動


In [ ]:
# TODO 🔴: 互動 Dashboard
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

# 讀資料
df_dash = pd.read_sql("SELECT * FROM cleaned_trips", conn)

# TODO: 建立下拉選單


# TODO: 定義更新函式


# TODO: 綁定互動


> 🅱️ `app.py` 是 solution。本地：`streamlit run app.py`


---
## Section 10（回家作業）：本地部署指引

```bash
cd data/raw/topic_4
uvicorn api:app --reload --port 8000    # Terminal 1
streamlit run app.py                     # Terminal 2
```

| 現在 | 升級後 | 對應課程 |
|------|--------|---------|
| SQLite | MySQL / BigQuery | 資料庫模組 |
| 手動跑 | Airflow DAG | Airflow 模組 |
| 本地 Streamlit | Docker 容器化 | Docker 模組 |
| 本地開發 | GCP 雲端部署 | GCP 模組 |
